# VisiumHD sp-SVC Reconstruction Impact

A route-specific reconstruction-impact analysis with matched cluster complexity and parent-internal diversity.


## 1. Route semantics and carrier audit

A same-resolution diagnostic asks whether the reconstructed partition is finer. A separate matched-K comparison asks how many assignments change after complexity is controlled. **Evidence boundary:** these are paired representation and spatial-pattern descriptions, not mechanism, truth, or clinical evidence.


In [ ]:
import os
import warnings
from pathlib import Path
os.environ.setdefault("KMP_WARNINGS", "0")
os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from revise.analysis.basic.spatial_region import compute_rarefied_window_diversity
from revise.analysis.reconstruction_impact import (compute_spatial_impact, file_sha256, load_reconstruction_impact_config, run_partition_analysis, write_analysis_artifacts, write_partition_artifacts, write_spatial_artifacts)
ROOT = Path(os.environ.get("REVISE_REPOSITORY_ROOT", Path.cwd())).resolve()
if not (ROOT / "configs" / "analysis").is_dir():
    raise RuntimeError("Run from the REVISE repository root.")
sns.set_theme(style="white", context="notebook")
PARENTS = ("Fibroblast", "Mono_Macro", "T")
ANATOMY_COLORS = {"Tumor":"#d73027", "Normal":"#2c7bb6", "Interface":"#fdae61", "Other":"#d9d9d9"}
def coordinates(adata):
    return pd.DataFrame(np.asarray(adata.obsm["spatial"], dtype=float)[:, :2], index=adata.obs_names, columns=["x", "y"])
def save_figure(fig, stem):
    fig.savefig(OUTPUT_DIR / "figures" / f"{stem}.png", dpi=180, bbox_inches="tight")
    plt.show(); plt.close(fig)
def scatter_windows(ax, table, value, title, *, cmap="viridis", vmin=None, vmax=None):
    image=ax.scatter(table.window_x, table.window_y, c=table[value], s=9, linewidths=0, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal"); ax.invert_yaxis(); plt.colorbar(image, ax=ax, shrink=.72)
def anatomy_map(ax, anatomy, title):
    ax.scatter(anatomy.window_x, anatomy.window_y, c=anatomy.level1_region.map(ANATOMY_COLORS), s=9, linewidths=0)
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal"); ax.invert_yaxis()
def revealed_metrics(run, parent):
    windows=run["impact"].unit_assignments.loc[:, ["x","y","window_id"]]
    values=compute_rarefied_window_diversity(windows, pd.Series(parent,index=windows.index), run["impact"].unit_assignments["reconstructed_cluster"].astype(str), min_parent_units=run["impact"].scale_audit["min_parent_units"], n_draws=run["impact"].scale_audit["rarefaction_draws"], random_state=42)
    return values.merge(run["impact"].anatomy_windows.loc[:, ["window_id","level1_region"]], on="window_id", how="left", validate="one_to_one")


In [ ]:
CONFIG=load_reconstruction_impact_config(ROOT / "configs" / "analysis" / "reconstruction_impact_visiumhd_p1crc.yaml")
OUTPUT_DIR=Path(os.environ.get("REVISE_ANALYSIS_OUTPUT_ROOT", CONFIG["output"]["dir"]))
if not OUTPUT_DIR.is_absolute(): OUTPUT_DIR=ROOT / OUTPUT_DIR
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
# Set True to release both global and parent sampling limits.
USE_FULL_VISIUMHD_COHORT = False
VISIUMHD_SAMPLE_N_UNITS = 30_000
comparison_config=CONFIG["partition_change"]["comparisons"][0]
raw_context=ad.read_h5ad(ROOT / comparison_config["raw_h5ad"], backed="r")
recon_context=ad.read_h5ad(ROOT / comparison_config["reconstructed_spatial_h5ad"], backed="r")
LEVEL1=comparison_config["level1_column"]
full_coordinates=coordinates(raw_context); full_level1=raw_context.obs[LEVEL1].astype(str).copy(); reconstructed_ids=recon_context.obs_names.copy()
if not set(reconstructed_ids) <= set(raw_context.obs_names): raise ValueError("Every reconstructed observation must occur in Raw.")
rng=np.random.default_rng(CONFIG["partition_change"]["random_state"])
def sampled(ids, limit):
    return pd.Index(ids) if USE_FULL_VISIUMHD_COHORT or len(ids) <= limit else pd.Index(rng.choice(np.asarray(ids), size=limit, replace=False))
global_ids=sampled(reconstructed_ids, VISIUMHD_SAMPLE_N_UNITS)
raw_global=raw_context[global_ids].to_memory(); recon_global=recon_context[global_ids].to_memory()
GLOBAL_PARTITION=run_partition_analysis(raw_global,recon_global,level1_col=LEVEL1,route_kind="sp_svc",resolution_mode="level1_ari",resolution_candidates=CONFIG["partition_change"]["level1_resolution_candidates"],within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"],random_state=CONFIG["partition_change"]["random_state"],n_top_genes=CONFIG["partition_change"]["n_top_genes"])
GLOBAL_COMPARISON=GLOBAL_PARTITION.comparisons["raw_to_recon_expression"]
RUNS={}; input_rows=[{"role":"full_raw_level1_context","units":raw_context.n_obs,"genes":raw_context.n_vars,"paired":False},{"role":"global_analysis_cohort","units":len(global_ids),"genes":recon_context.n_vars,"paired":True,"sampling":"full" if USE_FULL_VISIUMHD_COHORT else "deterministic_same_id_sample"}]
for parent in PARENTS:
    parent_ids=sampled(reconstructed_ids[full_level1.replace({"Mono/Macro":"Mono_Macro"}).reindex(reconstructed_ids).eq(parent)], CONFIG["partition_change"]["parent_sample_n_units"])
    raw_parent=raw_context[parent_ids].to_memory(); recon_parent=recon_context[parent_ids].to_memory()
    partition=run_partition_analysis(raw_parent,recon_parent,level1_col=LEVEL1,route_kind="sp_svc",resolution_mode="fixed_within_level1",within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"],random_state=CONFIG["partition_change"]["random_state"],n_top_genes=CONFIG["partition_change"]["n_top_genes"])
    comparison=partition.comparisons["raw_to_recon_expression"]; assignments=comparison.assignments.set_index("unit_id")
    impact=compute_spatial_impact(full_coordinates=full_coordinates,full_level1_labels=full_level1,paired_coordinates=coordinates(raw_parent),raw_labels=assignments.raw_cluster,reconstructed_labels=assignments.recon_cluster,unit_changed=assignments.unit_changed,microns_per_coordinate=CONFIG["spatial_region"]["microns_per_coordinate"],candidate_window_sides_um=CONFIG["spatial_region"]["candidate_window_sides_um"],min_parent_units=CONFIG["spatial_region"]["min_parent_units"],rarefaction_draws=CONFIG["spatial_region"]["rarefaction_draws"],threshold_bootstraps=CONFIG["spatial_region"]["threshold_bootstraps"],**CONFIG["spatial_region"]["anatomy_region"])
    RUNS[parent]={"partition":partition,"comparison":comparison,"impact":impact}; input_rows.append({"role":f"{parent}_paired_cohort","units":len(parent_ids),"genes":recon_parent.n_vars,"paired":True})
    write_partition_artifacts(OUTPUT_DIR / "parents" / parent,partition); write_spatial_artifacts(OUTPUT_DIR / "parents" / parent,impact)
input_audit=pd.DataFrame(input_rows)
manifest={"analysis_contract_version":2,"route":"sp_svc","sampling":{"global":len(global_ids),"full_switch":USE_FULL_VISIUMHD_COHORT},"inputs":{"raw":{"path":str(comparison_config["raw_h5ad"]),"sha256":file_sha256(ROOT / comparison_config["raw_h5ad"])},"reconstructed":{"path":str(comparison_config["reconstructed_spatial_h5ad"]),"sha256":file_sha256(ROOT / comparison_config["reconstructed_spatial_h5ad"])}},"global_matched_k_status":GLOBAL_PARTITION.matched_cluster_status}
write_analysis_artifacts(OUTPUT_DIR,config=CONFIG,manifest=manifest,input_audit=input_audit); write_partition_artifacts(OUTPUT_DIR / "global",GLOBAL_PARTITION)
raw_context.file.close(); recon_context.file.close()


In [ ]:
display(input_audit.fillna("—"))
display(pd.DataFrame([CONFIG["spatial_region"]]).loc[:,["microns_per_coordinate","candidate_window_sides_um","min_parent_units","rarefaction_draws"]])


## 2. Partition complexity diagnostic

This diagnostic preserves the Raw resolution definition. It supports a statement about finer or reorganized partitions, but never supplies the matched-K headline change.


In [ ]:
rows=[]
for parent,run in RUNS.items():
    row=next(iter(run["partition"].complexity_comparisons.values())).summary.iloc[0]; rows.append({"scope":parent,"Raw K":row.n_raw_clusters,"Recon K":row.n_recon_clusters,"ARI":row.ARI})
if "GLOBAL_PARTITION" in globals():
    row=next(iter(GLOBAL_PARTITION.complexity_comparisons.values())).summary.iloc[0]; rows.insert(0,{"scope":"Global","Raw K":row.n_raw_clusters,"Recon K":row.n_recon_clusters,"ARI":row.ARI})
complexity_table=pd.DataFrame(rows); fig,ax=plt.subplots(figsize=(7,4.5)); complexity_table.set_index("scope")[["Raw K","Recon K"]].plot.bar(ax=ax,color=["#4c78a8","#f58518"]); ax.set(ylabel="Cluster count",title="Same-resolution complexity diagnostic"); save_figure(fig,"01_partition_complexity")


In [ ]:
display(complexity_table)
display(Markdown("A higher reconstructed K supports the structural statement that coarse structure persists while some smaller or ambiguous Raw clusters are split or reorganized."))


## 3. Matched-complexity change and Level1 localization

Hungarian matching is calculated once per matched-K comparison. Its unit-level calls are reused for Level1 proportions and spatial localization; no regional re-matching occurs.


In [ ]:
rows=[]
for parent,run in RUNS.items():
    row=run["comparison"].summary.iloc[0]; rows.append({"scope":parent,"paired units":row.n_units,"Raw K":row.n_raw_clusters,"Recon K":row.n_recon_clusters,"ST-unit change":row.st_unit_change_fraction,"balanced change":row.balanced_cluster_change,"ARI":row.ARI,"status":run["partition"].matched_cluster_status})
if "GLOBAL_COMPARISON" in globals():
    row=GLOBAL_COMPARISON.summary.iloc[0]; rows.insert(0,{"scope":"Global","paired units":row.n_units,"Raw K":row.n_raw_clusters,"Recon K":row.n_recon_clusters,"ST-unit change":row.st_unit_change_fraction,"balanced change":row.balanced_cluster_change,"ARI":row.ARI,"status":GLOBAL_PARTITION.matched_cluster_status})
change_table=pd.DataFrame(rows); fig,axes=plt.subplots(1,len(RUNS),figsize=(5*len(RUNS),4.5),squeeze=False)
for ax,(parent,run) in zip(axes[0],RUNS.items()):
    c=run["comparison"].contingency; sns.heatmap(c.div(c.sum(axis=1),axis=0).fillna(0),ax=ax,cmap="mako",cbar=False); ax.set(title=f"{parent}: matched-K contingency",xlabel="Recon",ylabel="Raw")
save_figure(fig,"02_matched_k_contingency")


In [ ]:
display(change_table.round(4))
display(Markdown("ST-unit change is headline evidence only when the matched-K status is `ok`; otherwise it remains an explicit audit result."))


In [ ]:
tables=[]
for parent,run in RUNS.items():
    table=run["partition"].change_by_level1.copy(); table.insert(0,"scope",parent); tables.append(table)
if "GLOBAL_PARTITION" in globals():
    table=GLOBAL_PARTITION.change_by_level1.copy(); table.insert(0,"scope","Global"); tables.append(table)
level1_change=pd.concat(tables,ignore_index=True); plot=level1_change.loc[level1_change.level1.ne("Overall")]; fig,ax=plt.subplots(figsize=(9,4.5)); sns.barplot(data=plot,x="level1",y="change_fraction",hue="scope",ax=ax); ax.tick_params(axis="x",rotation=45); ax.set(xlabel="Raw Level1 label",ylabel="Changed-unit fraction",title="Level1 assignment change after global matching"); save_figure(fig,"03_level1_change_fraction")


In [ ]:
display(level1_change.loc[:,["scope","level1","total_units","changed_units","change_fraction","wilson_ci_lower","wilson_ci_upper","low_sample_size"]].sort_values(["scope","change_fraction"],ascending=[True,False]).head(12).round(4))
display(Markdown("The displayed rows are the largest changes; full Level1 tables, mapping and contingency artifacts remain available on disk."))


## 4. Anatomy overview

Tumor, Normal and Interface are fixed Level1 spatial context. They never tune K, scale or thresholds. This overview uses the Fibroblast grid; parent blocks retain their own occupancy-selected grids.


In [ ]:
anatomy=RUNS["Fibroblast"]["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"Level1 anatomy context"); save_figure(fig,"04_anatomy_overview")


In [ ]:
context=RUNS["Fibroblast"]["impact"].anatomy_context_summary
display(context.loc[context.level1_region.isin(["Tumor","Normal","Interface"]),["level1_region","full_level1_units","tissue_windows","area_mm2","area_fraction"]].round(4))


## 5. Fibroblast internal diversity and Regions

**Raw-Level1 / Recon subtype / Recon−1** is a state view: Raw is intentionally one parent label. **Raw-Leiden / Recon / Delta** uses the matched-K partitions and is the only row used for reconstruction-associated diversity change and Gain Region.


In [ ]:
RUN=RUNS["Fibroblast"]
REVEALED=revealed_metrics(RUN,"Fibroblast")
MATCHED=RUN["impact"].window_metrics
summary=RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"parent":"Fibroblast","paired units":summary.n_units,"Raw K":summary.n_raw_clusters,"Recon K":summary.n_recon_clusters,"ST-unit change":summary.st_unit_change_fraction,"ARI":summary.ARI,"window side (um)":RUN["impact"].scale_audit["main_window_side_um"],"valid windows":int(MATCHED.valid_window.sum())}]).round(4))
display(Markdown(f"Matched-K status: **{RUN['partition'].matched_cluster_status}**. Each valid window is paired-rarefied to four parent units."))


### Revealed subtype diversity

Visualization first: a state description of reconstruction-revealed parent-internal richness.


In [ ]:
valid=REVEALED.loc[REVEALED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],REVEALED,"neff_raw","Raw Level1 Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],REVEALED,"neff_recon","Recon subtype Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],REVEALED,"delta_neff","Recon minus 1",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"05_fibroblast_revealed_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"median Raw-Level1 Neff":valid.neff_raw.median(),"median Recon subtype Neff":valid.neff_recon.median(),"median Recon minus 1":valid.delta_neff.median()}]).round(3))
display(Markdown(f"Raw-Level1 Neff is {valid.neff_raw.median():.2f} by construction; this row describes revealed subtype richness rather than a fair effect size."))


### Reconstruction-associated local diversity change

The matched-K row holds units, grid, support and paired random draws constant.


In [ ]:
valid=MATCHED.loc[MATCHED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],MATCHED,"neff_raw","Raw-Leiden Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],MATCHED,"neff_recon","Recon Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],MATCHED,"delta_neff","Recon minus Raw",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"05_fibroblast_matched_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"Raw Neff median [Q1, Q3]":f"{valid.neff_raw.median():.2f} [{valid.neff_raw.quantile(.25):.2f}, {valid.neff_raw.quantile(.75):.2f}]","Recon Neff median [Q1, Q3]":f"{valid.neff_recon.median():.2f} [{valid.neff_recon.quantile(.25):.2f}, {valid.neff_recon.quantile(.75):.2f}]","Delta median [Q1, Q3]":f"{valid.delta_neff.median():.2f} [{valid.delta_neff.quantile(.25):.2f}, {valid.delta_neff.quantile(.75):.2f}]"}]))
display(Markdown(f"Matched-K local diversity changes by a median of **{valid.delta_neff.median():.2f}** effective clusters across valid windows."))


### Scale and data-driven Region thresholds

Window scale comes only from occupancy; State and Gain have separate bootstrap breakpoints.


In [ ]:
support=RUN["impact"].support_sensitivity
fig,ax=plt.subplots(figsize=(7,4.5)); ax.plot(support.window_side_length,support.retained_parent_unit_fraction,marker="o"); ax.axvline(RUN["impact"].scale_audit["main_window_side_um"],color="black",linestyle="--",label="selected"); ax.set(xlabel="Window side (um)",ylabel="Retained parent-unit fraction",title="Occupancy-only scale selection"); ax.legend(); save_figure(fig,"05_fibroblast_scale_knee")


In [ ]:
thresholds=pd.DataFrame([{"Region":"State",**RUN["impact"].state_threshold},{"Region":"Gain",**RUN["impact"].gain_threshold}])
display(thresholds.loc[:,["Region","status","threshold","ci_lower","ci_upper","n_windows","n_valid_bootstrap"]].round(3))
display(Markdown("State uses reconstructed Neff; Gain uses positive matched-K Delta Neff. A non-stable breakpoint intentionally creates no binary Region mask."))


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"State Region on anatomy"); state=MATCHED.loc[MATCHED.in_state_region.fillna(False).astype(bool)]; ax.scatter(state.window_x,state.window_y,c="#542788",s=14,linewidths=0); save_figure(fig,"05_fibroblast_state_region")


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"Gain Region on anatomy"); gain=MATCHED.loc[MATCHED.in_gain_region.fillna(False).astype(bool)]; ax.scatter(gain.window_x,gain.window_y,c="#1b7837",s=14,linewidths=0); save_figure(fig,"05_fibroblast_gain_region")


In [ ]:
extent=pd.concat([RUN["impact"].region_extent_by_anatomy.assign(Region="State"),RUN["impact"].gain_region_extent_by_anatomy.assign(Region="Gain")])
display(extent.loc[extent.level1_region.isin(["Overall","Tumor","Normal","Interface"]),["Region","level1_region","valid_windows","region_windows","region_area_mm2","area_fraction","unit_fraction","region_available"]].round(4))
display(Markdown("State and Gain masks are descriptive spatial windows, not pathological leading edge or validated anatomical components."))


## 6. Mono_Macro internal diversity and Regions

**Raw-Level1 / Recon subtype / Recon−1** is a state view: Raw is intentionally one parent label. **Raw-Leiden / Recon / Delta** uses the matched-K partitions and is the only row used for reconstruction-associated diversity change and Gain Region.


In [ ]:
RUN=RUNS["Mono_Macro"]
REVEALED=revealed_metrics(RUN,"Mono_Macro")
MATCHED=RUN["impact"].window_metrics
summary=RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"parent":"Mono_Macro","paired units":summary.n_units,"Raw K":summary.n_raw_clusters,"Recon K":summary.n_recon_clusters,"ST-unit change":summary.st_unit_change_fraction,"ARI":summary.ARI,"window side (um)":RUN["impact"].scale_audit["main_window_side_um"],"valid windows":int(MATCHED.valid_window.sum())}]).round(4))
display(Markdown(f"Matched-K status: **{RUN['partition'].matched_cluster_status}**. Each valid window is paired-rarefied to four parent units."))


### Revealed subtype diversity

Visualization first: a state description of reconstruction-revealed parent-internal richness.


In [ ]:
valid=REVEALED.loc[REVEALED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],REVEALED,"neff_raw","Raw Level1 Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],REVEALED,"neff_recon","Recon subtype Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],REVEALED,"delta_neff","Recon minus 1",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"06_mono-macro_revealed_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"median Raw-Level1 Neff":valid.neff_raw.median(),"median Recon subtype Neff":valid.neff_recon.median(),"median Recon minus 1":valid.delta_neff.median()}]).round(3))
display(Markdown(f"Raw-Level1 Neff is {valid.neff_raw.median():.2f} by construction; this row describes revealed subtype richness rather than a fair effect size."))


### Reconstruction-associated local diversity change

The matched-K row holds units, grid, support and paired random draws constant.


In [ ]:
valid=MATCHED.loc[MATCHED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],MATCHED,"neff_raw","Raw-Leiden Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],MATCHED,"neff_recon","Recon Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],MATCHED,"delta_neff","Recon minus Raw",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"06_mono-macro_matched_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"Raw Neff median [Q1, Q3]":f"{valid.neff_raw.median():.2f} [{valid.neff_raw.quantile(.25):.2f}, {valid.neff_raw.quantile(.75):.2f}]","Recon Neff median [Q1, Q3]":f"{valid.neff_recon.median():.2f} [{valid.neff_recon.quantile(.25):.2f}, {valid.neff_recon.quantile(.75):.2f}]","Delta median [Q1, Q3]":f"{valid.delta_neff.median():.2f} [{valid.delta_neff.quantile(.25):.2f}, {valid.delta_neff.quantile(.75):.2f}]"}]))
display(Markdown(f"Matched-K local diversity changes by a median of **{valid.delta_neff.median():.2f}** effective clusters across valid windows."))


### Scale and data-driven Region thresholds

Window scale comes only from occupancy; State and Gain have separate bootstrap breakpoints.


In [ ]:
support=RUN["impact"].support_sensitivity
fig,ax=plt.subplots(figsize=(7,4.5)); ax.plot(support.window_side_length,support.retained_parent_unit_fraction,marker="o"); ax.axvline(RUN["impact"].scale_audit["main_window_side_um"],color="black",linestyle="--",label="selected"); ax.set(xlabel="Window side (um)",ylabel="Retained parent-unit fraction",title="Occupancy-only scale selection"); ax.legend(); save_figure(fig,"06_mono-macro_scale_knee")


In [ ]:
thresholds=pd.DataFrame([{"Region":"State",**RUN["impact"].state_threshold},{"Region":"Gain",**RUN["impact"].gain_threshold}])
display(thresholds.loc[:,["Region","status","threshold","ci_lower","ci_upper","n_windows","n_valid_bootstrap"]].round(3))
display(Markdown("State uses reconstructed Neff; Gain uses positive matched-K Delta Neff. A non-stable breakpoint intentionally creates no binary Region mask."))


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"State Region on anatomy"); state=MATCHED.loc[MATCHED.in_state_region.fillna(False).astype(bool)]; ax.scatter(state.window_x,state.window_y,c="#542788",s=14,linewidths=0); save_figure(fig,"06_mono-macro_state_region")


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"Gain Region on anatomy"); gain=MATCHED.loc[MATCHED.in_gain_region.fillna(False).astype(bool)]; ax.scatter(gain.window_x,gain.window_y,c="#1b7837",s=14,linewidths=0); save_figure(fig,"06_mono-macro_gain_region")


In [ ]:
extent=pd.concat([RUN["impact"].region_extent_by_anatomy.assign(Region="State"),RUN["impact"].gain_region_extent_by_anatomy.assign(Region="Gain")])
display(extent.loc[extent.level1_region.isin(["Overall","Tumor","Normal","Interface"]),["Region","level1_region","valid_windows","region_windows","region_area_mm2","area_fraction","unit_fraction","region_available"]].round(4))
display(Markdown("State and Gain masks are descriptive spatial windows, not pathological leading edge or validated anatomical components."))


## 7. T internal diversity and Regions

**Raw-Level1 / Recon subtype / Recon−1** is a state view: Raw is intentionally one parent label. **Raw-Leiden / Recon / Delta** uses the matched-K partitions and is the only row used for reconstruction-associated diversity change and Gain Region.


In [ ]:
RUN=RUNS["T"]
REVEALED=revealed_metrics(RUN,"T")
MATCHED=RUN["impact"].window_metrics
summary=RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"parent":"T","paired units":summary.n_units,"Raw K":summary.n_raw_clusters,"Recon K":summary.n_recon_clusters,"ST-unit change":summary.st_unit_change_fraction,"ARI":summary.ARI,"window side (um)":RUN["impact"].scale_audit["main_window_side_um"],"valid windows":int(MATCHED.valid_window.sum())}]).round(4))
display(Markdown(f"Matched-K status: **{RUN['partition'].matched_cluster_status}**. Each valid window is paired-rarefied to four parent units."))


### Revealed subtype diversity

Visualization first: a state description of reconstruction-revealed parent-internal richness.


In [ ]:
valid=REVEALED.loc[REVEALED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],REVEALED,"neff_raw","Raw Level1 Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],REVEALED,"neff_recon","Recon subtype Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],REVEALED,"delta_neff","Recon minus 1",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"07_t_revealed_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"median Raw-Level1 Neff":valid.neff_raw.median(),"median Recon subtype Neff":valid.neff_recon.median(),"median Recon minus 1":valid.delta_neff.median()}]).round(3))
display(Markdown(f"Raw-Level1 Neff is {valid.neff_raw.median():.2f} by construction; this row describes revealed subtype richness rather than a fair effect size."))


### Reconstruction-associated local diversity change

The matched-K row holds units, grid, support and paired random draws constant.


In [ ]:
valid=MATCHED.loc[MATCHED.valid_window]; upper=max(1.,float(valid[["neff_raw","neff_recon"]].max().max())); bound=max(.05,float(np.nanmax(np.abs(valid.delta_neff))))
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
scatter_windows(axes[0],MATCHED,"neff_raw","Raw-Leiden Neff",vmin=1,vmax=upper)
scatter_windows(axes[1],MATCHED,"neff_recon","Recon Neff",vmin=1,vmax=upper)
scatter_windows(axes[2],MATCHED,"delta_neff","Recon minus Raw",cmap="coolwarm",vmin=-bound,vmax=bound)
save_figure(fig,"07_t_matched_diversity")


In [ ]:
display(pd.DataFrame([{"valid windows":int(valid.shape[0]),"Raw Neff median [Q1, Q3]":f"{valid.neff_raw.median():.2f} [{valid.neff_raw.quantile(.25):.2f}, {valid.neff_raw.quantile(.75):.2f}]","Recon Neff median [Q1, Q3]":f"{valid.neff_recon.median():.2f} [{valid.neff_recon.quantile(.25):.2f}, {valid.neff_recon.quantile(.75):.2f}]","Delta median [Q1, Q3]":f"{valid.delta_neff.median():.2f} [{valid.delta_neff.quantile(.25):.2f}, {valid.delta_neff.quantile(.75):.2f}]"}]))
display(Markdown(f"Matched-K local diversity changes by a median of **{valid.delta_neff.median():.2f}** effective clusters across valid windows."))


### Scale and data-driven Region thresholds

Window scale comes only from occupancy; State and Gain have separate bootstrap breakpoints.


In [ ]:
support=RUN["impact"].support_sensitivity
fig,ax=plt.subplots(figsize=(7,4.5)); ax.plot(support.window_side_length,support.retained_parent_unit_fraction,marker="o"); ax.axvline(RUN["impact"].scale_audit["main_window_side_um"],color="black",linestyle="--",label="selected"); ax.set(xlabel="Window side (um)",ylabel="Retained parent-unit fraction",title="Occupancy-only scale selection"); ax.legend(); save_figure(fig,"07_t_scale_knee")


In [ ]:
thresholds=pd.DataFrame([{"Region":"State",**RUN["impact"].state_threshold},{"Region":"Gain",**RUN["impact"].gain_threshold}])
display(thresholds.loc[:,["Region","status","threshold","ci_lower","ci_upper","n_windows","n_valid_bootstrap"]].round(3))
display(Markdown("State uses reconstructed Neff; Gain uses positive matched-K Delta Neff. A non-stable breakpoint intentionally creates no binary Region mask."))


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"State Region on anatomy"); state=MATCHED.loc[MATCHED.in_state_region.fillna(False).astype(bool)]; ax.scatter(state.window_x,state.window_y,c="#542788",s=14,linewidths=0); save_figure(fig,"07_t_state_region")


In [ ]:
anatomy=RUN["impact"].anatomy_windows; fig,ax=plt.subplots(figsize=(7,6)); anatomy_map(ax,anatomy,"Gain Region on anatomy"); gain=MATCHED.loc[MATCHED.in_gain_region.fillna(False).astype(bool)]; ax.scatter(gain.window_x,gain.window_y,c="#1b7837",s=14,linewidths=0); save_figure(fig,"07_t_gain_region")


In [ ]:
extent=pd.concat([RUN["impact"].region_extent_by_anatomy.assign(Region="State"),RUN["impact"].gain_region_extent_by_anatomy.assign(Region="Gain")])
display(extent.loc[extent.level1_region.isin(["Overall","Tumor","Normal","Interface"]),["Region","level1_region","valid_windows","region_windows","region_area_mm2","area_fraction","unit_fraction","region_available"]].round(4))
display(Markdown("State and Gain masks are descriptive spatial windows, not pathological leading edge or validated anatomical components."))


## 8. Cross-parent summary and evidence boundary

The compact table follows the evidence chain: matched-K assignment change, continuous local diversity, then independently thresholded State/Gain Region extent.


In [ ]:
rows=[]
for parent,run in RUNS.items():
    comparison=run["comparison"].summary.iloc[0]; metrics=run["impact"].window_metrics.loc[lambda frame:frame.valid_window]; state=run["impact"].region_extent_by_anatomy.set_index("level1_region").loc["Overall"]; gain=run["impact"].gain_region_extent_by_anatomy.set_index("level1_region").loc["Overall"]
    rows.append({"parent":parent,"ST-unit change":comparison.st_unit_change_fraction,"ARI":comparison.ARI,"median Delta Neff":metrics.delta_neff.median(),"State area fraction":state.area_fraction,"Gain area fraction":gain.area_fraction,"matched-K status":run["partition"].matched_cluster_status})
summary=pd.DataFrame(rows); display(summary.round(4)); display(Markdown("State/Gain masks are data-driven window masks. They are not direct evidence for pathological leading edge, anatomical components, or biological validity."))
